# ML Metadata Lab - Tracking the Iris Prediction Pipeline

**Hrishikesh Prabhu | IE-7374 MLOps | Spring 2026**

In my FastAPI lab, I built an Iris Flower Prediction API using Random Forest with confidence scores. This notebook tracks that entire pipeline using a custom metadata store built with SQLite — from dataset to trained model to API deployment.

**How my labs connect:**
- **GitHub Lab 1**: Learned testing and CI/CD
- **FastAPI Lab**: Built the prediction API with Random Forest and confidence scores
- **This Lab**: Tracking metadata of that same pipeline

In [4]:
import sqlite3
import pandas as pd
import numpy as np
import json
import os
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import joblib

print("All imports successful!")


All imports successful!



**Cell 3 (M):**

## Step 1: Build a Custom Metadata Store
Since ml-metadata doesn't support Python 3.13, we build our own using SQLite. This covers the same concepts — artifact types, execution types, events, contexts, and lineage tracking.

In [6]:
class MetadataStore:
    def __init__(self):
        self.conn = sqlite3.connect(":memory:")
        self.cursor = self.conn.cursor()
        self.cursor.executescript('''
            CREATE TABLE artifact_types (id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT UNIQUE, properties TEXT);
            CREATE TABLE artifacts (id INTEGER PRIMARY KEY AUTOINCREMENT, type_id INTEGER, uri TEXT, name TEXT, properties TEXT, created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP);
            CREATE TABLE execution_types (id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT UNIQUE, properties TEXT);
            CREATE TABLE executions (id INTEGER PRIMARY KEY AUTOINCREMENT, type_id INTEGER, state TEXT, properties TEXT, created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP);
            CREATE TABLE events (id INTEGER PRIMARY KEY AUTOINCREMENT, artifact_id INTEGER, execution_id INTEGER, event_type TEXT, created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP);
            CREATE TABLE contexts (id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT, description TEXT, owner TEXT);
            CREATE TABLE context_links (id INTEGER PRIMARY KEY AUTOINCREMENT, context_id INTEGER, artifact_id INTEGER, execution_id INTEGER);
        ''')
        self.conn.commit()

    def put_artifact_type(self, name, props):
        self.cursor.execute("INSERT OR IGNORE INTO artifact_types (name, properties) VALUES (?, ?)", (name, json.dumps(props)))
        self.conn.commit()
        self.cursor.execute("SELECT id FROM artifact_types WHERE name=?", (name,))
        return self.cursor.fetchone()[0]

    def put_artifact(self, type_id, uri, name, properties):
        self.cursor.execute("INSERT INTO artifacts (type_id, uri, name, properties) VALUES (?, ?, ?, ?)", (type_id, uri, name, json.dumps(properties)))
        self.conn.commit()
        return self.cursor.lastrowid

    def put_execution_type(self, name, props):
        self.cursor.execute("INSERT OR IGNORE INTO execution_types (name, properties) VALUES (?, ?)", (name, json.dumps(props)))
        self.conn.commit()
        self.cursor.execute("SELECT id FROM execution_types WHERE name=?", (name,))
        return self.cursor.fetchone()[0]

    def put_execution(self, type_id, state, properties=None):
        self.cursor.execute("INSERT INTO executions (type_id, state, properties) VALUES (?, ?, ?)", (type_id, state, json.dumps(properties or {})))
        self.conn.commit()
        return self.cursor.lastrowid

    def update_execution(self, exec_id, state):
        self.cursor.execute("UPDATE executions SET state=? WHERE id=?", (state, exec_id))
        self.conn.commit()

    def put_event(self, artifact_id, execution_id, event_type):
        self.cursor.execute("INSERT INTO events (artifact_id, execution_id, event_type) VALUES (?, ?, ?)", (artifact_id, execution_id, event_type))
        self.conn.commit()

    def put_context(self, name, description, owner):
        self.cursor.execute("INSERT INTO contexts (name, description, owner) VALUES (?, ?, ?)", (name, description, owner))
        self.conn.commit()
        return self.cursor.lastrowid

    def link_to_context(self, context_id, artifact_id=None, execution_id=None):
        self.cursor.execute("INSERT INTO context_links (context_id, artifact_id, execution_id) VALUES (?, ?, ?)", (context_id, artifact_id, execution_id))
        self.conn.commit()

    def get_artifacts(self):
        self.cursor.execute("SELECT a.id, at.name, a.uri, a.name, a.properties FROM artifacts a JOIN artifact_types at ON a.type_id=at.id")
        return self.cursor.fetchall()

    def get_executions(self):
        self.cursor.execute("SELECT e.id, et.name, e.state, e.properties FROM executions e JOIN execution_types et ON e.type_id=et.id")
        return self.cursor.fetchall()

    def get_events_by_artifact(self, artifact_id):
        self.cursor.execute("SELECT * FROM events WHERE artifact_id=?", (artifact_id,))
        return self.cursor.fetchall()

    def get_events_by_execution(self, execution_id):
        self.cursor.execute("SELECT * FROM events WHERE execution_id=?", (execution_id,))
        return self.cursor.fetchall()

    def get_artifact_by_id(self, art_id):
        self.cursor.execute("SELECT a.id, at.name, a.uri, a.name, a.properties FROM artifacts a JOIN artifact_types at ON a.type_id=at.id WHERE a.id=?", (art_id,))
        return self.cursor.fetchone()

    def trace_lineage(self, artifact_id):
        events = self.get_events_by_artifact(artifact_id)
        lineage = []
        for event in events:
            if event[3] == "OUTPUT":
                exec_events = self.get_events_by_execution(event[2])
                for ee in exec_events:
                    if ee[3] == "INPUT":
                        lineage.append(self.get_artifact_by_id(ee[1]))
        return lineage

store = MetadataStore()
print("Custom Metadata Store initialized!")

Custom Metadata Store initialized!


```

---

**Cell 5 (M):**
```
## Step 2: Prepare the Iris Dataset (Same as FastAPI Lab)

In [7]:
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
species_names = {0: "Setosa", 1: "Versicolor", 2: "Virginica"}

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

os.makedirs('pipeline_data/train', exist_ok=True)
os.makedirs('pipeline_data/eval', exist_ok=True)

train_df = pd.DataFrame(X_train, columns=feature_names)
train_df['species'] = [species_names[l] for l in y_train]
train_df['species_id'] = y_train
train_df.to_csv('pipeline_data/train/iris_train.csv', index=False)

eval_df = pd.DataFrame(X_test, columns=feature_names)
eval_df['species'] = [species_names[l] for l in y_test]
eval_df['species_id'] = y_test
eval_df.to_csv('pipeline_data/eval/iris_eval.csv', index=False)

print(f"Training: {len(train_df)} samples | Eval: {len(eval_df)} samples")
print(f"Features: {feature_names}")
train_df.head()


Training: 105 samples | Eval: 45 samples
Features: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species,species_id
0,5.5,2.4,3.7,1.0,Versicolor,1
1,6.3,2.8,5.1,1.5,Virginica,2
2,6.4,3.1,5.5,1.8,Virginica,2
3,6.6,3.0,4.4,1.4,Versicolor,1
4,7.2,3.6,6.1,2.5,Virginica,2


```

---

**Cell 7 (M):**
```
## Step 3: Register Artifact and Execution Types

In [8]:
dataset_type_id = store.put_artifact_type("IrisDataset", ["name", "split", "num_samples", "version"])
schema_type_id = store.put_artifact_type("DataSchema", ["name", "num_columns", "version"])
model_type_id = store.put_artifact_type("IrisModel", ["name", "algorithm", "accuracy", "version"])
api_type_id = store.put_artifact_type("APIConfig", ["name", "framework", "endpoints", "version"])

validation_type_id = store.put_execution_type("DataValidation", ["state"])
training_type_id = store.put_execution_type("ModelTraining", ["state", "algorithm"])
deploy_type_id = store.put_execution_type("APIDeployment", ["state", "framework"])

print("Artifact Types:", dataset_type_id, schema_type_id, model_type_id, api_type_id)
print("Execution Types:", validation_type_id, training_type_id, deploy_type_id)


Artifact Types: 1 2 3 4
Execution Types: 1 2 3


```

---

**Cell 9 (M):**
```
## Step 4: Data Validation — Generate Custom Schema
Instead of TFDV (not available on Mac), we build our own schema using pandas.

In [10]:
# Register dataset artifact
train_artifact_id = store.put_artifact(dataset_type_id, './pipeline_data/train/iris_train.csv', 
    'Iris Training Data', {"split": "train", "num_samples": len(train_df), "version": 1})

# Start validation
val_execution_id = store.put_execution(validation_type_id, "RUNNING")
store.put_event(train_artifact_id, val_execution_id, "INPUT")
print(f"Dataset ID: {train_artifact_id} | Validation ID: {val_execution_id} (RUNNING)")

# Generate custom schema
df = pd.read_csv('./pipeline_data/train/iris_train.csv')
schema = {"dataset": "Iris Training Data", "columns": {}}
for col in df.columns:
    info = {"dtype": str(df[col].dtype), "nulls": int(df[col].isnull().sum()), "unique": int(df[col].nunique())}
    if df[col].dtype in ['float64', 'int64']:
        info.update({"min": round(float(df[col].min()), 4), "max": round(float(df[col].max()), 4),
                     "mean": round(float(df[col].mean()), 4), "std": round(float(df[col].std()), 4)})
    elif df[col].dtype == 'object':
        info["categories"] = list(df[col].unique())
    schema["columns"][col] = info

with open('./pipeline_schema.json', 'w') as f:
    json.dump(schema, f, indent=2)

print("\nGenerated Schema:")
for col, info in schema["columns"].items():
    print(f"  {col}: {info['dtype']} | nulls={info['nulls']} | unique={info['unique']}")

# Register schema artifact and complete
schema_artifact_id = store.put_artifact(schema_type_id, './pipeline_schema.json', 
    'Iris Data Schema', {"num_columns": len(schema["columns"]), "version": 1})
store.put_event(schema_artifact_id, val_execution_id, "OUTPUT")
store.update_execution(val_execution_id, "COMPLETED")
print(f"\nSchema ID: {schema_artifact_id} | Validation: COMPLETED")


Dataset ID: 1 | Validation ID: 1 (RUNNING)

Generated Schema:
  sepal length (cm): float64 | nulls=0 | unique=34
  sepal width (cm): float64 | nulls=0 | unique=21
  petal length (cm): float64 | nulls=0 | unique=39
  petal width (cm): float64 | nulls=0 | unique=22
  species: str | nulls=0 | unique=3
  species_id: int64 | nulls=0 | unique=3

Schema ID: 2 | Validation: COMPLETED


```

---

**Cell 11 (M):**
```
## Step 5: Train the Model (Same as FastAPI Lab)
Random Forest with confidence scores — identical to our API.

In [11]:
train_execution_id = store.put_execution(training_type_id, "RUNNING", {"algorithm": "RandomForest"})
store.put_event(train_artifact_id, train_execution_id, "INPUT")

rf_model = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
avg_conf = np.mean(np.max(rf_model.predict_proba(X_test), axis=1))

os.makedirs('pipeline_model', exist_ok=True)
joblib.dump(rf_model, 'pipeline_model/iris_rf.pkl')

print(f"Accuracy: {acc*100:.2f}% | Avg Confidence: {avg_conf*100:.2f}%")
print(classification_report(y_test, y_pred, target_names=["Setosa", "Versicolor", "Virginica"]))

model_artifact_id = store.put_artifact(model_type_id, './pipeline_model/iris_rf.pkl',
    'Iris Random Forest v1', {"algorithm": "RandomForest", "accuracy": round(acc, 4), "version": 1})
store.put_event(model_artifact_id, train_execution_id, "OUTPUT")
store.update_execution(train_execution_id, "COMPLETED")
print(f"Model ID: {model_artifact_id} | Training: COMPLETED")


Accuracy: 100.00% | Avg Confidence: 96.94%
              precision    recall  f1-score   support

      Setosa       1.00      1.00      1.00        19
  Versicolor       1.00      1.00      1.00        13
   Virginica       1.00      1.00      1.00        13

    accuracy                           1.00        45
   macro avg       1.00      1.00      1.00        45
weighted avg       1.00      1.00      1.00        45

Model ID: 3 | Training: COMPLETED


```

---

**Cell 13 (M):**
```
## Step 6: Record API Deployment (Links to FastAPI Lab)
Tracking deployment of our FastAPI service with confidence scores and species descriptions.

In [12]:
deploy_execution_id = store.put_execution(deploy_type_id, "RUNNING", {"framework": "FastAPI"})
store.put_event(model_artifact_id, deploy_execution_id, "INPUT")

api_config = {
    "title": "Iris Flower Prediction API", "version": "2.0.0",
    "endpoints": [
        {"method": "GET", "path": "/", "desc": "Health check"},
        {"method": "POST", "path": "/predict", "desc": "Predict with confidence"},
        {"method": "GET", "path": "/model-info", "desc": "Model details"},
        {"method": "GET", "path": "/species", "desc": "Species info"}
    ],
    "features": ["confidence_scores", "species_descriptions"]
}
with open('./api_config.json', 'w') as f:
    json.dump(api_config, f, indent=2)

api_artifact_id = store.put_artifact(api_type_id, './api_config.json',
    'Iris Prediction API v2.0', {"framework": "FastAPI", "endpoints": 4, "version": 1})
store.put_event(api_artifact_id, deploy_execution_id, "OUTPUT")
store.update_execution(deploy_execution_id, "COMPLETED")
print(f"API ID: {api_artifact_id} | Deployment: COMPLETED")


API ID: 4 | Deployment: COMPLETED


```

---

**Cell 15 (M):**
```
## Step 7: Create Context (Group the Entire Pipeline)

In [13]:
context_id = store.put_context("IrisFlowerPipeline_v1",
    "End-to-end pipeline: data -> schema -> model -> API", "Hrishikesh Prabhu")

for art_id in [train_artifact_id, schema_artifact_id, model_artifact_id, api_artifact_id]:
    store.link_to_context(context_id, artifact_id=art_id)
for exec_id in [val_execution_id, train_execution_id, deploy_execution_id]:
    store.link_to_context(context_id, execution_id=exec_id)

print(f"Context ID: {context_id} — All linked!")


Context ID: 1 — All linked!


```

---

**Cell 17 (M):**
```
## Step 8: Query Metadata Store — Full Lineage

In [15]:


print("\n--- Artifacts ---")
for a in store.get_artifacts():
    print(f"  [{a[1]}] {a[3]} -> {a[2]}")

print("\n--- Executions ---")
for e in store.get_executions():
    print(f"  [{e[1]}] State: {e[2]}")

print("\n--- Lineage: API -> Model -> Dataset ---")
for model in store.trace_lineage(api_artifact_id):
    print(f"  API uses model: {model[3]}")
    for dataset in store.trace_lineage(model[0]):
        print(f"  Model trained on: {dataset[3]} ({dataset[2]})")



--- Artifacts ---
  [IrisDataset] Iris Training Data -> ./pipeline_data/train/iris_train.csv
  [DataSchema] Iris Data Schema -> ./pipeline_schema.json
  [IrisModel] Iris Random Forest v1 -> ./pipeline_model/iris_rf.pkl
  [APIConfig] Iris Prediction API v2.0 -> ./api_config.json

--- Executions ---
  [DataValidation] State: COMPLETED
  [ModelTraining] State: COMPLETED
  [APIDeployment] State: COMPLETED

--- Lineage: API -> Model -> Dataset ---
  API uses model: Iris Random Forest v1
  Model trained on: Iris Training Data (./pipeline_data/train/iris_train.csv)


```

---

**Cell 19 (M):**
```
## How All My Labs Connect

| Lab | What I Did | Connection |
|---|---|---|
| **GitHub Lab 1** | Calculator toolkit + pytest + GitHub Actions | Learned testing and CI/CD |
| **FastAPI Lab** | Iris API with Random Forest, confidence scores, species info | Applied testing to API endpoints |
| **MLMD Lab** | Tracked the same Iris pipeline with custom metadata store | Full lineage from dataset to deployed API |

In production, you need to know which dataset trained which model, and which model is served by which API. This lab demonstrates that lineage tracking.